<a href="https://colab.research.google.com/github/RAHUUL45879/Healthcare_Insurance_Claims_Analytics/blob/main/notebooks/healthcare-revenue-dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Healthcare Insurance Clamis Dashboard

In [ ]:
## import panda as pd

# install panda if not installed

!pip install pandas openpyxl

!pip install pyngrok

!pip install streamlit pandas plotly openpyxl pyngrok

!pip install streamlit


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 91.7 MB/s eta 0:00:00


In [ ]:
%%writefile dashboard.py

import streamlit as st
import pandas as pd
import plotly.express as px
import os

# ---------------------------
# PAGE CONFIG
# ---------------------------
st.set_page_config(
    page_title="Healthcare Insurance Claims Analytics Dashboard",
    layout="wide"
)

st.title("Healthcare Insurance Claims Analytics Dashboard")

# ---------------------------
# FILE UPLOADER
# ---------------------------
uploaded_file = st.file_uploader(
    "Upload Insurance Claims File",
    type=["csv", "xlsx", "xls", "xlsm"]
)

# ---------------------------
# PROCESS FILE
# ---------------------------
if uploaded_file:

    try:

        # ---------------------------
        # LOAD FILE
        # ---------------------------
        if uploaded_file.name.endswith(".csv"):
            df = pd.read_csv(uploaded_file)
        else:
            df = pd.read_excel(uploaded_file, engine="openpyxl")

        # ---------------------------
        # CLEAN COLUMN NAMES
        # ---------------------------
        df.columns = (
            df.columns
            .str.strip()
            .str.upper()
            .str.replace(" ", "_")
        )

        # ---------------------------
        # REQUIRED COLUMNS
        # ---------------------------
        required_cols = [
            'DATE_OF_SERVICE',
            'SUBMITTED_AMOUNT',
            'INSURANCE_PLAN_NAME',
            'DOCTOR_NAME'
        ]

        missing_cols = [
            col for col in required_cols
            if col not in df.columns
        ]

        if missing_cols:
            st.error(f"Missing required columns: {missing_cols}")
            st.stop()

        # ---------------------------
        # DATE CLEANING
        # ---------------------------
        df['DATE_OF_SERVICE'] = pd.to_datetime(
            df['DATE_OF_SERVICE'],
            errors='coerce'
        )

        df = df.dropna(subset=['DATE_OF_SERVICE'])

        # ---------------------------
        # AMOUNT COLUMNS
        # ---------------------------
        amount_cols = [
            'SUBMITTED_AMOUNT',
            'PAID_AMOUNT',
            'DENIED_AMOUNT',
            'RESUBMITTED_AMOUNT_1',
            'RESUBMISSION_PAID_AMOUNT_1',
            'RESUBMISSION_DENIED_AMOUNT_REMITTANCE_1',
            'RESUBMITTED_AMOUNT2',
            'RESUBMISSION_PAID_AMOUNT2',
            'RESUBMISSION_DENIED_AMOUNT_REMITTANCE_2'
        ]

        # Create missing columns if absent
        for col in amount_cols:
            if col not in df.columns:
                df[col] = 0

        # Convert to numeric
        df[amount_cols] = (
            df[amount_cols]
            .apply(pd.to_numeric, errors='coerce')
            .fillna(0)
        )

        # ---------------------------
        # DERIVED DATE COLUMNS
        # ---------------------------
        df['YEAR'] = df['DATE_OF_SERVICE'].dt.year
        df['MONTH'] = df['DATE_OF_SERVICE'].dt.strftime('%b')
        df['QUARTER'] = df['DATE_OF_SERVICE'].dt.quarter

        # ---------------------------
        # KPI CALCULATIONS
        # ---------------------------
        df['TOTAL_SUBMITTED_AMOUNT'] = round(
            df['SUBMITTED_AMOUNT']
            + df['RESUBMITTED_AMOUNT_1']
            + df['RESUBMITTED_AMOUNT2'],
            2
        )

        df['TOTAL_PAID_AMOUNT'] = round(
            df['PAID_AMOUNT']
            + df['RESUBMISSION_PAID_AMOUNT_1']
            + df['RESUBMISSION_PAID_AMOUNT2'],
            2
        )

        df['TOTAL_DENIED_AMOUNT'] = round(
            df['DENIED_AMOUNT']
            + df['RESUBMISSION_DENIED_AMOUNT_REMITTANCE_1']
            + df['RESUBMISSION_DENIED_AMOUNT_REMITTANCE_2'],
            2
        )

        df['TOTAL_PENDING_AMOUNT'] = round(
            df['TOTAL_SUBMITTED_AMOUNT']
            - (
                df['TOTAL_PAID_AMOUNT']
                + df['TOTAL_DENIED_AMOUNT']
            ),
            2
        )

        # ---------------------------
        # SIDEBAR FILTERS
        # ---------------------------
        st.sidebar.header("Filters")

        # Doctor Filter
        doctor_options = ["ALL"] + sorted(
            list(df['DOCTOR_NAME'].dropna().unique())
        )

        selected_doctor = st.sidebar.multiselect(
            "Select Doctor",
            options=doctor_options,
            default=["ALL"]
        )

        # Year Filter
        year_options = ["ALL"] + sorted(
            list(df['YEAR'].dropna().unique())
        )

        selected_year = st.sidebar.multiselect(
            "Select Year",
            options=year_options,
            default=["ALL"]
        )

        # Insurance Filter
        insurance_options = ["ALL"] + sorted(
            list(df['INSURANCE_PLAN_NAME'].dropna().unique())
        )

        selected_insurance = st.sidebar.multiselect(
            "Select Insurance Plan",
            options=insurance_options,
            default=["ALL"]
        )

        # ---------------------------
        # FILTER LOGIC
        # ---------------------------
        effective_doctor = (
            df['DOCTOR_NAME'].unique()
            if "ALL" in selected_doctor
            else selected_doctor
        )

        effective_year = (
            df['YEAR'].unique()
            if "ALL" in selected_year
            else selected_year
        )

        effective_insurance = (
            df['INSURANCE_PLAN_NAME'].unique()
            if "ALL" in selected_insurance
            else selected_insurance
        )

        filtered_df = df[
            (df['DOCTOR_NAME'].isin(effective_doctor)) &
            (df['YEAR'].isin(effective_year)) &
            (df['INSURANCE_PLAN_NAME'].isin(effective_insurance))
        ]

        # ---------------------------
        # KPI METRICS
        # ---------------------------
        total_claimed = filtered_df['TOTAL_SUBMITTED_AMOUNT'].sum()
        total_paid = filtered_df['TOTAL_PAID_AMOUNT'].sum()
        total_denied = filtered_df['TOTAL_DENIED_AMOUNT'].sum()
        total_pending = filtered_df['TOTAL_PENDING_AMOUNT'].sum()

        col1, col2, col3, col4 = st.columns(4)

        col1.metric("Total Claimed", f"{total_claimed:,.2f}")
        col2.metric("Total Paid", f"{total_paid:,.2f}")
        col3.metric("Total Denied", f"{total_denied:,.2f}")
        col4.metric("Total Pending", f"{total_pending:,.2f}")

        # ---------------------------
        # SUMMARY TABLE
        # ---------------------------
        grouped_summary = filtered_df.groupby(
            ['YEAR', 'INSURANCE_PLAN_NAME']
        ).agg(
            Claimed_Amount=('TOTAL_SUBMITTED_AMOUNT', 'sum'),
            Paid_Amount=('TOTAL_PAID_AMOUNT', 'sum'),
            Denied_Amount=('TOTAL_DENIED_AMOUNT', 'sum'),
            Pending_Amount=('TOTAL_PENDING_AMOUNT', 'sum')
        ).reset_index()

        st.subheader("Insurance Summary Table")
        st.dataframe(grouped_summary)

        # ---------------------------
        # MONTHLY SUBMISSION TABLE
        # ---------------------------
        month_order = [
            'Jan', 'Feb', 'Mar', 'Apr',
            'May', 'Jun', 'Jul', 'Aug',
            'Sep', 'Oct', 'Nov', 'Dec'
        ]

        grouped_submitted = (
            filtered_df.groupby(
                ['YEAR', 'INSURANCE_PLAN_NAME', 'MONTH']
            )['SUBMITTED_AMOUNT']
            .sum()
            .unstack()
            .fillna(0)
        )

        grouped_submitted = grouped_submitted.reindex(
            columns=month_order,
            fill_value=0
        )

        st.subheader("Monthly Submitted Amount")
        st.dataframe(grouped_submitted)

        # ---------------------------
        # VISUALIZATIONS
        # ---------------------------
        st.subheader("Dashboard Visualizations")

        # Doctor Summary
        doctor_summary = filtered_df.groupby(
            'DOCTOR_NAME'
        ).agg(
            Claimed=('TOTAL_SUBMITTED_AMOUNT', 'sum'),
            Paid=('TOTAL_PAID_AMOUNT', 'sum'),
            Denied=('TOTAL_DENIED_AMOUNT', 'sum'),
            Pending=('TOTAL_PENDING_AMOUNT', 'sum')
        ).reset_index()

        # Bar Chart
        bar_chart = px.bar(
            doctor_summary,
            x='DOCTOR_NAME',
            y=['Claimed', 'Paid', 'Denied', 'Pending'],
            barmode='group',
            title='Doctor-wise Claims Performance'
        )

        st.plotly_chart(bar_chart, use_container_width=True)

        # Insurance Chart
        insurance_summary = filtered_df.groupby(
            'INSURANCE_PLAN_NAME'
        ).agg(
            Claimed=('TOTAL_SUBMITTED_AMOUNT', 'sum'),
            Paid=('TOTAL_PAID_AMOUNT', 'sum')
        ).reset_index()

        insurance_chart = px.bar(
            insurance_summary,
            x='INSURANCE_PLAN_NAME',
            y=['Claimed', 'Paid'],
            barmode='group',
            title='Insurance Plan Performance'
        )

        st.plotly_chart(insurance_chart, use_container_width=True)

        # Scatter Chart
        performance_df = filtered_df.groupby(
            'DOCTOR_NAME'
        ).agg(
            Total_Claims=('CLAIM_ID', 'count'),
            Total_Paid=('TOTAL_PAID_AMOUNT', 'sum')
        ).reset_index()

        scatter_chart = px.scatter(
            performance_df,
            x='Total_Claims',
            y='Total_Paid',
            color='DOCTOR_NAME',
            size='Total_Claims',
            title='Doctor Performance Analysis'
        )

        st.plotly_chart(scatter_chart, use_container_width=True)

        # Histogram
        hist_chart = px.histogram(
            filtered_df,
            x='MONTH',
            y='SUBMITTED_AMOUNT',
            color='YEAR',
            category_orders={'MONTH': month_order},
            title='Monthly Claims Distribution'
        )

        st.plotly_chart(hist_chart, use_container_width=True)

        # ---------------------------
        # EXPORT TO EXCEL
        # ---------------------------
        output_file = "Insurance_Claims_Report.xlsx"

        with pd.ExcelWriter(
            output_file,
            engine='openpyxl'
        ) as writer:

            grouped_summary.to_excel(
                writer,
                sheet_name='Summary',
                index=False
            )

            grouped_submitted.to_excel(
                writer,
                sheet_name='Monthly_Submission',
                index=True
            )

            filtered_df.to_excel(
                writer,
                sheet_name='Filtered_Raw_Data',
                index=False
            )

        # ---------------------------
        # DOWNLOAD BUTTON
        # ---------------------------
        with open(output_file, "rb") as file:
            st.download_button(
                label="Download Excel Report",
                data=file,
                file_name="Insurance_Claims_Report.xlsx"
            )

        # Cleanup
        os.remove(output_file)

    except Exception as e:
        st.error(f"Error: {e}")

Writing dashboard.py


In [ ]:
%%writefile dashboard.py
import streamlit as st
import pandas as pd
import plotly.express as px
import os  # For file cleanup

st.set_page_config(page_title="Insurance Claims Dashboard", layout="wide")
st.title("Insurance Claims Analysis and Visualization Dashboard")

# File uploader with size warning
uploaded_file = st.file_uploader("Upload Excel/CSV File (Max 50MB recommended)", type=["csv", "xls", "xlsx", "xlsm"])

if uploaded_file:
    try:
        # Load data
        if uploaded_file.name.endswith(".csv"):
            df = pd.read_csv(uploaded_file)
        else:
            df = pd.read_excel(uploaded_file, engine='openpyxl')

        # Basic validation: Check for required columns
        required_cols = ['RA_RECEIVE_DATE', 'INSURANCE / TPA', 'PAID_AMOUNT']
        missing_cols = [col for col in required_cols if col not in df.columns]
        if missing_cols:
            st.error(f"Missing required columns: {missing_cols}. Please check your file.")
            st.stop()

        # Data cleaning
        df.columns = df.columns.str.strip()

        # Convert amount columns to numeric (expanded for robustness)
        amount_cols = ['PAID_AMOUNT', 'ReSubmission_Paid_Amount_1', 'ReSubmission_Paid_Amount2',
                       'SUBMITTED_AMOUNT', 'ReSubmitted_Amount_1', 'ReSubmitted_Amount2',
                       'DENIED_AMOUNT', 'RESUBMISSION_DENIED_AMOUNT_RA_1', 'RESUBMISSION_DENIED_AMOUNT_RA_2']

        df[amount_cols] = df[amount_cols].apply(pd.to_numeric, errors='coerce').fillna(0)

        # Convert 'RA_RECEIVE_DATE' column to datetime
        df['RA_RECEIVE_DATE'] = pd.to_datetime(df['RA_RECEIVE_DATE'], errors='coerce')
        df = df.dropna(subset=['RA_RECEIVE_DATE'])

        # Extract year, month, and quarter
        df['RA_RECEIVE_Year'] = df['RA_RECEIVE_DATE'].dt.year
        df['RA_RECEIVE_Month'] = df['RA_RECEIVE_DATE'].dt.strftime('%b')
        df['Quarter'] = df['RA_RECEIVE_DATE'].dt.quarter

        # Simplified calculations (adjust logic as needed for accuracy)
        df['Total Submitted Amount'] = round(df['SUBMITTED_AMOUNT'] + df['ReSubmitted_Amount_1'] + df['ReSubmitted_Amount2'], 2)
        df['Total Paid Amount'] = round(df['PAID_AMOUNT'] + df['ReSubmission_Paid_Amount_1'] + df['ReSubmission_Paid_Amount2'], 2)
        df['Total Denied Amount'] = round((df['DENIED_AMOUNT'] - df['ReSubmitted_Amount_1']) + (df['RESUBMISSION_DENIED_AMOUNT_RA_1'] - df['ReSubmitted_Amount2']) + df['RESUBMISSION_DENIED_AMOUNT_RA_2'], 2)
        df['Total Pending Amount'] = round(df['SUBMITTED_AMOUNT'] - (df['Total Paid Amount'] + df['Total Denied Amount']), 2)

        # Sidebar filters for interactivity
        st.sidebar.header("Filters")

        # Options for years with "ALL" added
        year_options = ["ALL"] + sorted(df['RA_RECEIVE_Year'].unique())
        selected_year = st.sidebar.multiselect("Select Year(s)", options=year_options, default=["ALL"])

        # Options for insurance with "ALL" added
        insurance_options = ["ALL"] + list(df['INSURANCE / TPA'].unique())
        selected_insurance = st.sidebar.multiselect("Select Insurance(s)", options=insurance_options, default=["ALL"])

        # Determine effective selections: if "ALL" is selected, use all options; otherwise, use selected ones
        effective_year = df['RA_RECEIVE_Year'].unique() if "ALL" in selected_year else [y for y in selected_year if y != "ALL"]
        effective_insurance = df['INSURANCE / TPA'].unique() if "ALL" in selected_insurance else [i for i in selected_insurance if i != "ALL"]

        # Apply filters using the effective selections
        filtered_df = df[
            (df['RA_RECEIVE_Year'].isin(effective_year)) &
            (df['INSURANCE / TPA'].isin(effective_insurance))
        ]

        # Group by year, insurance provider, and month (filtered)
        month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
        grouped_paid = filtered_df.groupby(['RA_RECEIVE_Year', 'INSURANCE / TPA', 'RA_RECEIVE_Month'])['Total Paid Amount'].sum().unstack(fill_value=0)
        grouped_paid = grouped_paid.reindex(columns=month_order, fill_value=0).reset_index().sort_values(by='RA_RECEIVE_Year')

        # Additional summary table (new for completeness)
        summary_table = filtered_df.groupby(['RA_RECEIVE_Year', 'INSURANCE / TPA']).agg(
            Total_Submitted=('Total Submitted Amount', 'sum'),
            Total_Paid=('Total Paid Amount', 'sum'),
            Total_Denied=('Total Denied Amount', 'sum'),
            Total_Pending=('Total Pending Amount', 'sum')
        ).reset_index().sort_values(by='RA_RECEIVE_Year')

        # Display Tables
        st.subheader("Paid Amount Per Month (Filtered)")
        st.dataframe(grouped_paid)
        st.subheader("Summary Table: Submitted, Paid, Denied by Year and Insurance (Filtered)")
        st.dataframe(summary_table)

        # Charts Section
        st.subheader("Data Visualizations (Filtered Data)")
        if filtered_df.empty:
            st.warning("No data available for the selected filters. Please adjust your selections.")
        else:
            # Existing/Enhanced Charts
            st.markdown("### Trends and Comparisons")

            # Enhanced Bar Chart: Total Paid Amount per Year (added color by quarter for more insight)
            yearly_paid = filtered_df.groupby(['RA_RECEIVE_Year', 'Quarter'])['Total Paid Amount'].sum().reset_index()
            bar_fig = px.bar(yearly_paid, x='RA_RECEIVE_Year', y='Total Paid Amount', color='Quarter',
                             title="Yearly Paid Amount by Quarter", labels={'Total Paid Amount': "Total Paid ($)"},
                             color_discrete_sequence=px.colors.qualitative.Set1)
            st.plotly_chart(bar_fig)

            # Enhanced Bar Chart: Paid Amount by Insurance Provider (horizontal for readability)
            insurance_paid = filtered_df.groupby('INSURANCE / TPA')['Total Paid Amount'].sum().reset_index()
            bar_insurance = px.bar(insurance_paid, x='Total Paid Amount', y='INSURANCE / TPA', orientation='h',
                                   title="Paid Amount by Insurance Provider",
                                   labels={'Total Paid Amount': "Total Paid ($)"}, color_discrete_sequence=px.colors.qualitative.Set2)
            st.plotly_chart(bar_insurance)

            # Enhanced Histogram: Distribution of Paid Amounts (added marginal rug plot)
            hist_fig = px.histogram(filtered_df, x='Total Paid Amount', nbins=50,
                                    title="Distribution of Paid Amounts", labels={'Total Paid Amount': "Paid Amount ($)"},
                                    marginal="rug", color_discrete_sequence=['#1f77b4'])
            st.plotly_chart(hist_fig)

            # Enhanced Scatter Chart: Paid vs. Submitted by Insurance (added trendline)
            scatter_data = filtered_df.groupby('INSURANCE / TPA').agg(
                Total_Submitted=('Total Submitted Amount', 'sum'),
                Total_Paid=('Total Paid Amount', 'sum')
            ).reset_index()
            scatter_fig = px.scatter(scatter_data, x='Total_Submitted', y='Total_Paid', color='INSURANCE / TPA',
                                     size='Total_Paid', title="Insurance Performance: Submitted vs. Paid Amounts",
                                     labels={'Total_Submitted': 'Total Submitted ($)', 'Total_Paid': 'Total Paid ($)'},
                                     trendline="ols")
            st.plotly_chart(scatter_fig)


            st.markdown("### Time-Series and Trends")

            # Line Chart: Paid Amount Trends Over Time
            time_trend = filtered_df.groupby(['RA_RECEIVE_Year', 'RA_RECEIVE_Month'])['Total Paid Amount'].sum().reset_index()
            time_trend['Month-Year'] = time_trend['RA_RECEIVE_Month'] + '-' + time_trend['RA_RECEIVE_Year'].astype(str)
            line_fig = px.line(time_trend, x='Month-Year', y='Total Paid Amount',
                               title="Monthly Paid Amount Trends", labels={'Total Paid Amount': "Total Paid ($)"},
                               color_discrete_sequence=['#ff7f0e'])
            st.plotly_chart(line_fig)

            # Area Chart: Cumulative Paid Amounts by Insurance
            area_data = filtered_df.groupby('INSURANCE / TPA')['Total Paid Amount'].sum().reset_index().sort_values('Total Paid Amount', ascending=False)
            area_fig = px.area(area_data, x='INSURANCE / TPA', y='Total Paid Amount',
                               title="Cumulative Paid Amounts by Insurance Provider",
                               labels={'Total Paid Amount': "Total Paid ($)"}, color_discrete_sequence=px.colors.qualitative.Pastel)
            st.plotly_chart(area_fig)

            st.markdown("### Proportions and Distributions")

            # Pie Chart: Paid vs. Denied Proportions by Insurance
            pie_data = filtered_df.groupby('INSURANCE / TPA').agg(
                Total_Paid=('Total Paid Amount', 'sum'),
                Total_Denied=('Total Denied Amount', 'sum')
            ).reset_index()
            pie_data_melted = pie_data.melt(id_vars='INSURANCE / TPA', value_vars=['Total_Paid', 'Total_Denied'],
                                             var_name='Status', value_name='Amount')
            pie_fig = px.pie(pie_data_melted, values='Amount', names='Status', color='Status',
                             title="Paid vs. Denied Amounts by Insurance Provider",
                             labels={'Amount': 'Amount ($)'}, color_discrete_map={'Total_Paid': '00F7FF', 'Total_Denied': 'red'})
            st.plotly_chart(pie_fig)

            # Box Plot: Paid Amount Distribution by Insurance
            box_fig = px.box(filtered_df, x='INSURANCE / TPA', y='Total Paid Amount',
                             title="Paid Amount Distribution by Insurance Provider",
                             labels={'Total Paid Amount': "Paid Amount ($)"}, color='INSURANCE / TPA')
            st.plotly_chart(box_fig)

            st.markdown("### Advanced Insights")

            # Heatmap: Monthly Paid Amounts by Year and Insurance
            heatmap_data = filtered_df.groupby(['RA_RECEIVE_Year', 'RA_RECEIVE_Month', 'INSURANCE / TPA'])['Total Paid Amount'].sum().reset_index()
            heatmap_pivot = heatmap_data.pivot_table(values='Total Paid Amount', index=['RA_RECEIVE_Year', 'INSURANCE / TPA'], columns='RA_RECEIVE_Month', fill_value=0)
            heatmap_pivot = heatmap_pivot.reindex(columns=month_order, fill_value=0)
            heatmap_fig = px.imshow(heatmap_pivot, text_auto=True, aspect="auto",
                                    title="Heatmap of Monthly Paid Amounts by Year and Insurance",
                                    labels=dict(x="Month", y="Year & Insurance", color="Paid Amount ($)"))
            st.plotly_chart(heatmap_fig)

            # Scatter Plot: Paid vs. Denied by Insurance
            scatter_denied = filtered_df.groupby('INSURANCE / TPA').agg(
                Total_Paid=('Total Paid Amount', 'sum'),
                Total_Denied=('Total Denied Amount', 'sum')
            ).reset_index()
            scatter_denied_fig = px.scatter(scatter_denied, x='Total_Denied', y='Total_Paid', color='INSURANCE / TPA',
                                            size='Total_Paid', title="Paid vs. Denied Amounts by Insurance",
                                            labels={'Total_Denied': 'Total Denied ($)', 'Total_Paid': 'Total Paid ($)'})
            st.plotly_chart(scatter_denied_fig)

            # Stacked Bar Chart: Multi-Metric Breakdown by Year
            stacked_data = filtered_df.groupby('RA_RECEIVE_Year').agg(
                Total_Submitted=('Total Submitted Amount', 'sum'),
                Total_Paid=('Total Paid Amount', 'sum'),
                Total_Denied=('Total Denied Amount', 'sum'),
                Total_Pending=('Total Pending Amount', 'sum')
            ).reset_index()
            stacked_fig = px.bar(stacked_data, x='RA_RECEIVE_Year', y=['Total_Submitted', 'Total_Paid', 'Total_Denied', 'Total_Pending'],
                                 title="Submitted, Paid, Denied, and Pending Amounts by Year",
                                 labels={'value': 'Amount ($)', 'variable': 'Metric'}, barmode='stack')
            st.plotly_chart(stacked_fig)

            # Faceted Histogram: Paid Amounts by Quarter
            facet_hist_fig = px.histogram(filtered_df, x='Total Paid Amount', facet_col='Quarter', nbins=30,
                                          title="Distribution of Paid Amounts by Quarter",
                                          labels={'Total Paid Amount': "Paid Amount ($)"})
            st.plotly_chart(facet_hist_fig)

        # Export to Excel (filtered data, multiple sheets)
        output_file = "Insurance_Claims_Report.xlsx"
        with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
            grouped_paid.to_excel(writer, sheet_name="Paid Claims Per Month", index=False)
            summary_table.to_excel(writer, sheet_name="Summary", index=False)
            filtered_df.to_excel(writer, sheet_name="All Claims Raw (Filtered)", index=False)

        # Download button for the Excel file
        with open(output_file, "rb") as file:
            st.download_button("Download Insurance Claims Report (Filtered)", file, file_name="Insurance_Claims_Report.xlsx")
        # Cleanup
        os.remove(output_file)

    except Exception as e:
        st.error(f"Error processing file: {e}. Please check your data format and try again.")

Overwriting dashboard.py


In [ ]:
!ngrok config add-authtoken 2v4XoShuteefvDNMNpLvNZvF9dM_338NZYBDPQfVKq1TYyt4F

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
from pyngrok import ngrok
import os

# Set your authtoken (Only run once in terminal, remove from script after that)
ngrok.set_auth_token("2v4XoShuteefvDNMNpLvNZvF9dM_338NZYBDPQfVKq1TYyt4F")

# Run Streamlit in the background
os.system("streamlit run dashboard.py &")

# Expose Streamlit using Ngrok
rahuldey_url = ngrok.connect("http://localhost:8501")
print(f"Streamlit is running on {rahuldey_url.public_url}")



Streamlit is running on https://ffbf-35-185-51-47.ngrok-free.app
